# CRM cleaning – `hotel_bookings_crm.csv` (Phase A)

Every cleaning step: **what**, **why**, **rows before → after**.

- **CHECK** cells show the numbers a decision is based on. They do not change the data.
- **Cleaning before the merge:** a step on a single file must not harm the merge, and must not delete information that the other file (PMS) can recompute or check.
- The notebook reads the raw file from `Project/data/raw` and writes the clean table to `Project/data/processed`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

RAW_PATH   = '../data/raw/hotel_bookings_crm.csv'
CLEAN_PATH = '../data/processed/crm_clean.csv'


In [2]:
# The file uses ';' as separator. We read everything as text (dtype=str)
# so that no column is converted wrongly on its own.
raw = pd.read_csv(RAW_PATH, sep=';', dtype=str)
df = raw.copy()     # 'raw' stays untouched for before/after comparisons
print(df.shape)
df.head()


(24608, 18)


,BookingRef,property,checkin_date,n_adults,lead_time_days,guest_country,market_segment,distribution_channel,customer_type,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,agent_id,company_id,days_in_waiting_list,reservation_status,reservation_status_date,total_price
0,BK310511,City Hotel,2016-12-03,2,73,USA,Online TA,TA/TO,Transient,0,0,0,7,NaN,0,Check-Out,2016-12-04,NaN
1,BK300310,City Hotel,2016-05-29,2,106 days,DEU,Offline TA/TO,TA/TO,Transient-Party,0,0,0,27,NaN,0,Check-Out,2016-06-01,"242,25"
2,BK302934,City Hotel,2017-07-13,3,356,NOR,Online TA,TA/TO,Transient,0,0,0,9,NaN,0,Canceled,2016-08-29,"529,2"
3,BK320441,Resort Hotel,2016-10-30,2,14,FRA,Online TA,TA/TO,Transient,0,0,0,241,NaN,0,Check-Out,2016-11-14,"799,8"
4,BK316173,City Hotel,2016-04-29,3,27,ITA,Online TA,TA/TO,Transient,0,0,0,9,NaN,0,Canceled,2016-04-04,"531,0"


## 2. Description and quality report (before cleaning)


In [3]:
# Overview: type of each column and how many values are not missing
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24608 entries, 0 to 24607
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   BookingRef                      24600 non-null  object
 1   property                        24600 non-null  object
 2   checkin_date                    24600 non-null  object
 3   n_adults                        24354 non-null  object
 4   lead_time_days                  24600 non-null  object
 5   guest_country                   24600 non-null  object
 6   market_segment                  24600 non-null  object
 7   distribution_channel            24600 non-null  object
 8   customer_type                   24600 non-null  object
 9   is_repeated_guest               24600 non-null  object
 10  previous_cancellations          24600 non-null  object
 11  previous_bookings_not_canceled  24600 non-null  object
 12  agent_id                        21280 non-null

In [4]:
# How many values are missing in each column, as a number and as a percentage
missing = df.isna().sum().to_frame('n_missing')
missing['pct'] = (missing['n_missing'] / len(df) * 100).round(2)
missing


,n_missing,pct
BookingRef,8,0.03
property,8,0.03
checkin_date,8,0.03
n_adults,254,1.03
lead_time_days,8,0.03
guest_country,8,0.03
market_segment,8,0.03
distribution_channel,8,0.03
customer_type,8,0.03
is_repeated_guest,8,0.03


In [5]:
# Is the same value written in different ways? (case, spaces, typos)
for c in ['property', 'market_segment', 'distribution_channel', 'customer_type', 'reservation_status']:
    print(f'--- {c} ---')
    print(df[c].value_counts(dropna=False), '\n')


--- property ---
property
City Hotel        15692
Resort Hotel       7924
City Hotel          173
CITY HOTEL          166
 City Hotel         163
city hotel          148
Resort Hotel         87
resort hotel         86
 Resort Hotel        82
RESORT HOTEL         79
NaN                   8
Name: count, dtype: int64 

--- market_segment ---
market_segment
Online TA        11592
Offline TA/TO     4846
Groups            4040
Direct            2484
Corporate         1080
Complementary      140
Online T.A.         81
Onlne TA            57
online ta           49
Aviation            47
Offline TA-TO       31
offline ta/to       24
Ofline TA/TO        24
Groups              23
Direct              18
direct              15
Group               15
groups              13
Corporte             9
NaN                  8
corporate            7
complementary        3
Undefined            2
Name: count, dtype: int64 

--- distribution_channel ---
distribution_channel
TA/TO        20217
Direct        2923

In [6]:
print('Fully duplicated rows:', df.duplicated().sum())
print('Repeated BookingRef:', df['BookingRef'].dropna().duplicated().sum())
print('Fully empty rows:', df.isna().all(axis=1).sum())


Fully duplicated rows: 107
Repeated BookingRef: 100
Fully empty rows: 8


In [7]:
# checkin_date is written in several formats: we count how many of each kind
d = df['checkin_date'].dropna()
print('yyyy-mm-dd :', d.str.contains(r'^\d{4}-\d{2}-\d{2}$', regex=True).sum())
print('dd/mm/yyyy :', d.str.contains(r'^\d{2}/\d{2}/\d{4}$', regex=True).sum())
print('dd Mon yyyy:', d.str.contains(r'^\d{1,2} [A-Za-z]{3} \d{4}$', regex=True).sum())


yyyy-mm-dd : 19867
dd/mm/yyyy : 3687
dd Mon yyyy: 1046


In [8]:
# lead_time_days should be only a number, but some values look like "106 days"
lead = df['lead_time_days'].dropna()
not_number = ~lead.str.contains(r'^\d+$', regex=True)
print('Non-numeric lead_time values:', not_number.sum())
print('Examples:', lead[not_number].head().tolist())

# total_price uses the comma as decimal separator
print('total_price values with a comma:', df['total_price'].dropna().str.contains(',').sum())


Non-numeric lead_time values: 982
Examples: ['106 days', '86 days', '1 days', '214 days', '186 days']
total_price values with a comma: 24106


## 3. Cleaning


### 3.1 Whitespace and empty rows
- **What:** we strip the spaces at the start and end of every cell and drop the completely empty rows.
- **Why:** `' City Hotel'` and `'City Hotel  '` are the same value; the empty rows contain no booking.


In [9]:
# Strip spaces at the start and end of every cell (all columns are text)
for col in df.columns:
    df[col] = df[col].str.strip()

# Drop the completely empty rows
n = len(df)
df = df.dropna(how='all')
print(f'Empty rows: {n} -> {len(df)} rows ({n - len(df)} removed)')


Empty rows: 24608 -> 24600 rows (8 removed)


### 3.2 Duplicates
- **What:** we drop identical rows.
- **Why:** every repeated `BookingRef` is an exact copy of the same booking; keeping it would inflate counts and revenue.
- After this step `BookingRef` must be unique, because it is the key for the merge.


In [10]:
# Drop identical rows (same booking exported twice)
n = len(df)
df = df.drop_duplicates()
print(f'Duplicates: {n} -> {len(df)} rows ({n - len(df)} removed)')

# Check: BookingRef must be unique and never empty (needed for the merge)
assert df['BookingRef'].notna().all()
assert df['BookingRef'].is_unique
print('BookingRef is unique:', df['BookingRef'].is_unique)


Duplicates: 24600 -> 24500 rows (100 removed)
BookingRef is unique: True


### 3.3 Categorical columns
- **What:** every variant (upper/lower case, typos, dashes) is brought back to one correct value.
- **Why:** otherwise a `groupby` counts `Canceled`, `CANCELED` and `Cancelled` as three different statuses.
- **`Undefined`** in `market_segment` and `distribution_channel` is a placeholder for "unknown", not a real category: it is impossible on its own, so it becomes NaN. Before changing it we create the flags `segment_undefined` and `channel_undefined`. No row is dropped.

In [11]:
# CHECK: how many rows have 'Undefined', before deciding (no changes)
print("market_segment = 'Undefined':", (df['market_segment'].str.lower() == 'undefined').sum())
print("distribution_channel = 'Undefined':", (df['distribution_channel'] == 'Undefined').sum())

market_segment = 'Undefined': 2
distribution_channel = 'Undefined': 3


In [12]:
# property: 10 variants -> 2 values ('City Hotel', 'Resort Hotel')
df['property'] = df['property'].str.title()
df['property'].value_counts(dropna=False)


property
City Hotel      16283
Resort Hotel     8217
Name: count, dtype: int64

In [13]:
# market_segment: first everything lower case, then we fix the typos
seg = df['market_segment'].str.lower()
seg = seg.replace({
    'online t.a.': 'online ta', 'onlne ta': 'online ta',
    'offline ta-to': 'offline ta/to', 'ofline ta/to': 'offline ta/to',
    'group': 'groups', 'corporte': 'corporate',
})
# Flag before changing: 'undefined' will become NaN
df['segment_undefined'] = seg == 'undefined'

# then every value goes back to the official spelling ('undefined' -> missing)
df['market_segment'] = seg.map({
    'online ta': 'Online TA', 'offline ta/to': 'Offline TA/TO', 'groups': 'Groups',
    'direct': 'Direct', 'corporate': 'Corporate', 'complementary': 'Complementary',
    'aviation': 'Aviation', 'undefined': np.nan,
})
print('Segments set to NaN:', df['segment_undefined'].sum())
df['market_segment'].value_counts(dropna=False)

Segments set to NaN: 2


market_segment
Online TA        11737
Offline TA/TO     4907
Groups            4071
Direct            2504
Corporate         1090
Complementary      142
Aviation            47
NaN                  2
Name: count, dtype: int64

In [14]:
# lower case and without spaces/dashes: 'Check-Out', 'CheckOut', 'check out' all become 'checkout'
status = df['reservation_status'].str.lower().str.replace(r'[\s-]', '', regex=True)
df['reservation_status'] = status.map({
    'checkout': 'Check-Out', 'canceled': 'Canceled', 'cancelled': 'Canceled', 'noshow': 'No-Show',
})
# Check: a variant missing from the map would become NaN
assert df['reservation_status'].notna().all(), 'Unmapped status variant'
df['reservation_status'].value_counts(dropna=False)


reservation_status
Check-Out    15326
Canceled      8918
No-Show        256
Name: count, dtype: int64

In [15]:
# 'Undefined' is not a real channel -> missing value (flag first)
df['channel_undefined'] = df['distribution_channel'] == 'Undefined'
df['distribution_channel'] = df['distribution_channel'].replace('Undefined', np.nan)
print('Channels set to NaN:', df['channel_undefined'].sum())
df['distribution_channel'].value_counts(dropna=False)

Channels set to NaN: 3


distribution_channel
TA/TO        20140
Direct        2905
Corporate     1402
GDS             50
NaN              3
Name: count, dtype: int64

### 3.4 Country codes
- **What:** everything upper case; `PT` and `Portugal` → `PRT` (same country written differently); `unknown` and `-` → NaN.
- **Why:** `unknown` and `-` are placeholders: `isna()` does not see them, but they are not real countries.
- **`CN` is not changed.** It is ambiguous: China already appears as `CHN` (217 rows), while Canada (`CAN`) never appears, so `CN` is probably Canada. It is not proven row by row and the PMS has no country, so the value stays `CN` and the rows are marked with the flag `country_cn_ambiguous`.
- Before changing we create the flag `country_placeholder`. No row is dropped.

In [16]:
# CHECK: values that are not 3-letter codes, and the codes close to 'CN' (no changes)
c = df['guest_country'].str.upper()
print("'CHN':", (c == 'CHN').sum(), "| 'CAN':", (c == 'CAN').sum())
c[c.str.len() != 3].value_counts(dropna=False)

'CHN': 217 | 'CAN': 0


guest_country
CN          228
UNKNOWN      61
-            53
PT           23
PORTUGAL     23
Name: count, dtype: int64

In [17]:
# Flag before changing (and a flag for 'CN', which stays unchanged)
country = df['guest_country'].str.upper()
df['country_placeholder'] = country.isin(['UNKNOWN', '-'])
df['country_cn_ambiguous'] = country == 'CN'

# Make the country codes uniform; placeholders become NaN; 'CN' is not touched
df['guest_country'] = country.replace({
    'PT': 'PRT', 'PORTUGAL': 'PRT',
    'UNKNOWN': np.nan, '-': np.nan,
})
print('Country placeholders set to NaN:', df['country_placeholder'].sum())
print("'CN' flagged as ambiguous (unchanged):", df['country_cn_ambiguous'].sum())

# Check: which values are not 3 letters long?
c = df['guest_country']
print('Non 3-letter codes:', c[c.str.len() != 3].value_counts(dropna=False).to_dict())

Country placeholders set to NaN: 114
'CN' flagged as ambiguous (unchanged): 228
Non 3-letter codes: {'CN': 228, nan: 114}


### 3.5 Dates
- **What:** `checkin_date` has 3 formats (`2016-12-03`, `03/12/2016`, `03 Dec 2016`): we read them one at a time with `pd.to_datetime(..., errors='coerce')` and join the results with `fillna`.
- **Why:** we need real dates (`datetime64`) to use `.dt` and compare dates with each other.


In [18]:
# Each format is read separately: what does not match becomes NaT
iso  = pd.to_datetime(df['checkin_date'], format='%Y-%m-%d', errors='coerce')
eu   = pd.to_datetime(df['checkin_date'], format='%d/%m/%Y', errors='coerce')
text = pd.to_datetime(df['checkin_date'], format='%d %b %Y', errors='coerce')
df['checkin_date'] = iso.fillna(eu).fillna(text)

df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'], format='%Y-%m-%d')

# Check: no date may stay unread
assert df['checkin_date'].notna().all(), 'Unparsed check-in dates'
print('Check-in range:', df['checkin_date'].min().date(), '->', df['checkin_date'].max().date())


Check-in range: 2015-07-01 -> 2017-08-31


### 3.6 Numeric columns
- **What:** we remove `" days"` from `lead_time_days`, replace the decimal comma with a dot in `total_price`, and convert the integer columns to numbers.
- **Why:** with text you cannot compute sums or means (`'1650' + '980'` = `'1650980'`).
- Columns with NaN (`n_adults`, `agent_id`, `company_id`) become `float64`: NaN is a float.


In [19]:
df['lead_time_days'] = pd.to_numeric(df['lead_time_days'].str.replace(' days', '', regex=False))
df['total_price']    = pd.to_numeric(df['total_price'].str.replace(',', '.', regex=False))

num_cols = ['n_adults', 'is_repeated_guest', 'previous_cancellations',
            'previous_bookings_not_canceled', 'agent_id', 'company_id', 'days_in_waiting_list']
for col in num_cols:
    df[col] = pd.to_numeric(df[col])

# Before correcting we look at min and max (describe shows the impossible extremes)
df[['n_adults', 'lead_time_days', 'total_price', 'days_in_waiting_list']].describe()


,n_adults,lead_time_days,total_price,days_in_waiting_list
count,24256.000000,24500.000000,24010.000000,24500.000000
mean,1.858550,113.810490,359.156165,2.267102
std,0.615799,333.236553,336.733011,17.430131
min,0.000000,0.000000,0.000000,0.000000
25%,2.000000,18.000000,148.985000,0.000000
50%,2.000000,70.000000,270.000000,0.000000
75%,2.000000,160.000000,450.000000,0.000000
max,55.000000,9999.000000,5400.000000,391.000000


### 3.7 Accuracy: impossible values
**CHECK** (cell below): how many suspicious values there are, before deciding.

**Decisions:**
- **`lead_time_days = 9999`**: it is a placeholder for "unknown", not a 27-year wait. It is impossible on its own → NaN, with `.mask()`. First we create the flag `lead_time_missing`.
- **`n_adults` above 10** (26 and 55): **no change in this phase.** The value is not impossible on its own, because real groups exist (both rows are `Group`). It is marked with `adults_over_10`; in Phase B it is compared with `adults` of the PMS, as planned in the PMS notebook.
- **`total_price`:** the IQR rule flags about 1,500 high prices, but they are long stays or groups (*"IQR flags long tails"*): they are real values, so they are kept.

In [20]:
# CHECK: one mask per check (no changes)
lead_placeholder = df['lead_time_days'] == 9999
print('lead_time = 9999:', lead_placeholder.sum(),
      '| highest other value:', df.loc[~lead_placeholder, 'lead_time_days'].max())

many = df['n_adults'] > 10
print('n_adults > 10:', many.sum(), '| values:', df.loc[many, 'n_adults'].tolist(),
      '| customer_type:', df.loc[many, 'customer_type'].tolist())

# total_price: IQR rule only to measure
q1, q3 = df['total_price'].quantile([0.25, 0.75])
high = q3 + 1.5 * (q3 - q1)
print(f'total_price above IQR fence ({high:.0f}):', (df['total_price'] > high).sum())
print('Negative prices:', (df['total_price'] < 0).sum())

lead_time = 9999: 25 | highest other value: 737
n_adults > 10: 2 | values: [26.0, 55.0] | customer_type: ['Group', 'Group']
total_price above IQR fence (902): 1504
Negative prices: 0


In [21]:
# lead_time = 9999 -> NaN (flag first)
df['lead_time_missing'] = df['lead_time_days'] == 9999
df['lead_time_days'] = df['lead_time_days'].mask(df['lead_time_missing'])
print('lead_time set to NaN:', df['lead_time_missing'].sum())

# n_adults > 10: no change, only the flag for Phase B
df['adults_over_10'] = df['n_adults'] > 10
print('n_adults > 10 flagged (unchanged):', df['adults_over_10'].sum())

lead_time set to NaN: 25
n_adults > 10 flagged (unchanged): 2


### 3.8 Consistency between columns
- **What:** the status date must agree with the status.
  - `Check-Out`: the guest leaves after arriving, so the status date cannot be *before* check-in.
  - `Canceled`: a booking is cancelled before the stay, so the status date cannot be *after* check-in (after arrival it would be a no-show or a check-out).
  - `No-Show`: it is recorded when the guest does not arrive, so the status date cannot be *before* check-in.
- **Move chosen:** *flag only*, because we do not know which of the two dates is wrong. We add the column `date_inconsistent` and decide after the comparison with the PMS arrival date in Phase B.

**CHECK** (cell below): how many rows break each rule.

In [22]:
# CHECK: one mask per status (no changes)
checkout_bad = (df['reservation_status'] == 'Check-Out') & \
               (df['reservation_status_date'] < df['checkin_date'])
canceled_bad = (df['reservation_status'] == 'Canceled') & \
               (df['reservation_status_date'] > df['checkin_date'])
noshow_bad   = (df['reservation_status'] == 'No-Show') & \
               (df['reservation_status_date'] < df['checkin_date'])
print('Check-Out dated before check-in:', checkout_bad.sum())
print('Canceled dated after check-in:  ', canceled_bad.sum())
print('No-Show dated before check-in:  ', noshow_bad.sum())


Check-Out dated before check-in: 37
Canceled dated after check-in:   50
No-Show dated before check-in:   1


In [23]:
# One flag for the three rules; the dates are not changed
df['date_inconsistent'] = checkout_bad | canceled_bad | noshow_bad
print('Rows flagged as date_inconsistent:', df['date_inconsistent'].sum())


Rows flagged as date_inconsistent: 88


## 4. Export


In [24]:
# DATASET PROFILE: crm_clean.csv
# Rows x columns : 24,500 x 25 (raw: 24,608 x 18)
# Removed        : 8 empty rows, 100 exact duplicates
# Fixed          : spellings (property, segment, status, country), 3 date formats,
#                  ' days' in lead_time, decimal comma in total_price
# Set to NaN     : 25 lead_time = 9999, 114 country placeholders,
#                  2 'Undefined' segments, 3 'Undefined' channels
# Flagged only   : 228 'CN' (China or Canada? left unchanged), 2 n_adults > 10,
#                  88 status dates inconsistent with check-in
# Open questions : n_adults > 10 and missing prices -> check against the PMS in Phase B

print('Raw shape:  ', raw.shape)
print('Clean shape:', df.shape)
print('\nMissing values after cleaning:')
print(df.isna().sum()[df.isna().sum() > 0])

# How many rows each flag marks
flags = ['segment_undefined', 'channel_undefined', 'country_placeholder', 'country_cn_ambiguous',
         'lead_time_missing', 'adults_over_10', 'date_inconsistent']
print('\nRows marked by each flag:')
print(df[flags].sum())

df.to_csv(CLEAN_PATH, index=False)
print(f'\nSaved clean CRM to {CLEAN_PATH}')

Raw shape:   (24608, 18)
Clean shape: (24500, 25)

Missing values after cleaning:
n_adults                  244
lead_time_days             25
guest_country             114
market_segment              2
distribution_channel        3
agent_id                 3300
company_id              23124
total_price               490
dtype: int64

Rows marked by each flag:
segment_undefined         2
channel_undefined         3
country_placeholder     114
country_cn_ambiguous    228
lead_time_missing        25
adults_over_10            2
date_inconsistent        88
dtype: int64

Saved clean CRM to ../data/processed/crm_clean.csv
